# 🌴 Arecanut Plant Disease Dataset Preparation in Google Colab
**Engineering Mini Project**: Detection of Diseases in Arecanut Plants using Deep Learning (MobileNetV2)

This notebook automates:
1. Kaggle API authentication via `kaggle.json`
2. Automated dataset download & extraction
3. Inspection of folder structures
4. Flattening nested directories into standardized class folders
5. Dataset distribution visualization (bar chart)
6. Cleaning: Corrupted image detection + MD5 duplicate removal
7. Stratified 70/15/15 Train/Validation/Test split with fixed seed
8. High-performance `tf.data` pipeline (`image_dataset_from_directory` + `cache()` + `prefetch()`)
9. 3x3 visual sample grid with class labels

In [ ]:
# ==============================================================================
# CELL 1: Install Kaggle CLI and Authenticate using kaggle.json
# ==============================================================================
!pip install -q kaggle

import os
import shutil
from google.colab import files

print("📥 Please upload your 'kaggle.json' file (downloaded from Kaggle -> Account -> Create New Token):")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    
    target_file = os.path.join(kaggle_dir, 'kaggle.json')
    if os.path.exists(target_file):
        os.remove(target_file)
    shutil.move('kaggle.json', target_file)
    os.chmod(target_file, 0o600)
    
    print("\n✅ kaggle.json configured successfully with 600 permissions!")
else:
    raise FileNotFoundError("❌ 'kaggle.json' was not uploaded. Please re-run this cell and upload it.")

In [ ]:
# ==============================================================================
# CELL 2: Download Dataset from Kaggle and Extract into /content/dataset_raw
# ==============================================================================
# 👉 REPLACE WITH YOUR KAGGLE DATASET SLUG (owner/dataset-name)
# Example: "siddharthsuresh/arecanut-disease-dataset" or "username/dataset-slug"
KAGGLE_DATASET_SLUG = "owner/dataset-name"   # <--- PASTE YOUR SLUG HERE

RAW_DIR = "/content/dataset_raw"
os.makedirs(RAW_DIR, exist_ok=True)

print(f"⏬ Downloading '{KAGGLE_DATASET_SLUG}' using Kaggle API...")
!kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p {RAW_DIR} --unzip

print(f"\n✅ Dataset downloaded and extracted to: {RAW_DIR}")

In [ ]:
# ==============================================================================
# CELL 3: Print Directory Structure (First 2 Levels)
# ==============================================================================
import os

def print_tree(start_path, max_depth=2):
    print(f"📂 Folder Structure for: {start_path}\n")
    base_depth = start_path.rstrip(os.sep).count(os.sep)
    
    for root, dirs, files in os.walk(start_path):
        depth = root.count(os.sep) - base_depth
        if depth > max_depth:
            continue
        indent = "   " * depth
        folder_name = os.path.basename(root)
        image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))]
        
        if depth == 0:
            print(f"📦 [{folder_name}] (Root)")
        else:
            print(f"{indent}├── 📁 {folder_name}/ ({len(image_files)} images, {len(dirs)} subfolders)")

print_tree(RAW_DIR, max_depth=2)

In [ ]:
# ==============================================================================
# CELL 4: Handle Nested Folders and Flatten into /content/dataset/<class_name>/
# ==============================================================================
import shutil
from collections import defaultdict

FLATTENED_DIR = "/content/dataset_flattened"
os.makedirs(FLATTENED_DIR, exist_ok=True)

# Check if dataset already has pre-split train/val/test folders
existing_subdirs = [d.lower() for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))]
HAS_PREDEFINED_SPLIT = any(s in existing_subdirs for s in ['train', 'training']) and any(s in existing_subdirs for s in ['test', 'val', 'validation'])

print(f"ℹ️ Predefined train/test split detected: {HAS_PREDEFINED_SPLIT}")

valid_exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
ignored_dirs = {'train', 'test', 'val', 'validation', 'dataset', 'data', 'images', 'image'}

flattened_counts = defaultdict(int)

if not HAS_PREDEFINED_SPLIT:
    for root, dirs, files in os.walk(RAW_DIR):
        images = [f for f in files if f.lower().endswith(valid_exts)]
        if not images:
            continue
        
        # Determine class name from parent directory name
        parent_dir = os.path.basename(root)
        class_name = parent_dir.strip().replace('_', ' ').title()
        
        target_class_dir = os.path.join(FLATTENED_DIR, class_name)
        os.makedirs(target_class_dir, exist_ok=True)
        
        for img_name in images:
            src_path = os.path.join(root, img_name)
            dest_name = f"{class_name[:3]}_{flattened_counts[class_name]:05d}_{img_name}"
            dest_path = os.path.join(target_class_dir, dest_name)
            shutil.copy2(src_path, dest_path)
            flattened_counts[class_name] += 1
            
    print(f"\n✅ Flattened all images into: {FLATTENED_DIR}")
else:
    print("ℹ️ Keeping original train/test folder structure for downstream pipeline.")
    FLATTENED_DIR = RAW_DIR

In [ ]:
# ==============================================================================
# CELL 5: Print Class Distribution & Plot Bar Chart
# ==============================================================================
import matplotlib.pyplot as plt

class_counts = {}
search_dir = FLATTENED_DIR

for entry in os.listdir(search_dir):
    entry_path = os.path.join(search_dir, entry)
    if os.path.isdir(entry_path) and entry.lower() not in ['train', 'val', 'test']:
        imgs = [f for f in os.listdir(entry_path) if f.lower().endswith(valid_exts)]
        class_counts[entry] = len(imgs)

print("📊 Class Distribution:")
print("=" * 45)
total_images = sum(class_counts.values())
for cls, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    pct = (count / total_images * 100) if total_images > 0 else 0
    print(f"  • {cls:<25}: {count:>5} images ({pct:>5.1f}%)")
print("=" * 45)
print(f"  Total Images: {total_images}")

# Plot Bar Chart
plt.figure(figsize=(10, 5))
bars = plt.bar(class_counts.keys(), class_counts.values(), color='#16a34a', edgecolor='#14532d', width=0.55)
plt.title('Arecanut Disease Dataset - Class Distribution', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Disease Class', fontsize=11, fontweight='bold')
plt.ylabel('Number of Images', fontsize=11, fontweight='bold')
plt.xticks(rotation=25, ha='right', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Add values on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + (total_images * 0.01), int(yval), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELL 6: Remove Corrupted Images & Duplicate Images (MD5 Hash Check)
# ==============================================================================
import hashlib
from PIL import Image

def calculate_md5(file_path):
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

print("🧹 Cleaning Dataset (Removing Corrupted & Duplicate Files)...")

seen_hashes = set()
corrupted_count = 0
duplicate_count = 0
valid_count = 0

for root, dirs, files in os.walk(FLATTENED_DIR):
    for file in files:
        if file.lower().endswith(valid_exts):
            file_path = os.path.join(root, file)
            
            # 1. Test image integrity using PIL
            is_valid = False
            try:
                with Image.open(file_path) as img:
                    img.verify()
                with Image.open(file_path) as img:
                    img.load()  # Verify pixel buffer
                is_valid = True
            except Exception:
                os.remove(file_path)
                corrupted_count += 1
                continue
            
            # 2. Check for duplicate images via MD5 hash
            if is_valid:
                file_hash = calculate_md5(file_path)
                if file_hash in seen_hashes:
                    os.remove(file_path)
                    duplicate_count += 1
                else:
                    seen_hashes.add(file_hash)
                    valid_count += 1

print("=" * 45)
print(f"✅ Cleaning Completed:")
print(f"   • Corrupted Images Removed: {corrupted_count}")
print(f"   • Duplicate Images Removed: {duplicate_count}")
print(f"   • Clean Valid Images Kept:  {valid_count}")
print("=" * 45)

In [ ]:
# ==============================================================================
# CELL 7: Stratified Train / Validation / Test Split (70 / 15 / 15)
# ==============================================================================
import random

FINAL_SPLIT_DIR = "/content/dataset_split"

if not HAS_PREDEFINED_SPLIT:
    TRAIN_RATIO = 0.70
    VAL_RATIO   = 0.15
    TEST_RATIO  = 0.15
    RANDOM_SEED = 42
    random.seed(RANDOM_SEED)
    
    for split in ['train', 'val', 'test']:
        for cls in class_counts.keys():
            os.makedirs(os.path.join(FINAL_SPLIT_DIR, split, cls), exist_ok=True)
            
    print(f"✂️ Splitting dataset (70% Train, 15% Val, 15% Test, Seed={RANDOM_SEED})...")
    
    for cls in class_counts.keys():
        src_class_dir = os.path.join(FLATTENED_DIR, cls)
        images = [f for f in os.listdir(src_class_dir) if f.lower().endswith(valid_exts)]
        random.shuffle(images)
        
        n_total = len(images)
        n_train = int(n_total * TRAIN_RATIO)
        n_val   = int(n_total * VAL_RATIO)
        
        train_imgs = images[:n_train]
        val_imgs   = images[n_train:n_train + n_val]
        test_imgs  = images[n_train + n_val:]
        
        for img in train_imgs:
            shutil.copy2(os.path.join(src_class_dir, img), os.path.join(FINAL_SPLIT_DIR, 'train', cls, img))
        for img in val_imgs:
            shutil.copy2(os.path.join(src_class_dir, img), os.path.join(FINAL_SPLIT_DIR, 'val', cls, img))
        for img in test_imgs:
            shutil.copy2(os.path.join(src_class_dir, img), os.path.join(FINAL_SPLIT_DIR, 'test', cls, img))
            
        print(f"   • {cls:<22} -> Train: {len(train_imgs):>4} | Val: {len(val_imgs):>4} | Test: {len(test_imgs):>4}")
        
    print(f"\n✅ Stratified dataset split successfully prepared at: {FINAL_SPLIT_DIR}")
else:
    print("ℹ️ Utilizing existing Train/Test dataset directories directly.")
    FINAL_SPLIT_DIR = RAW_DIR

In [ ]:
# ==============================================================================
# CELL 8: Load with tf.keras image_dataset_from_directory + Cache & Prefetch
# ==============================================================================
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_path = os.path.join(FINAL_SPLIT_DIR, 'train')
val_path   = os.path.join(FINAL_SPLIT_DIR, 'val') if os.path.exists(os.path.join(FINAL_SPLIT_DIR, 'val')) else os.path.join(FINAL_SPLIT_DIR, 'test')

print("🚀 Loading Datasets using tf.keras.utils.image_dataset_from_directory...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
    seed=SEED
)

class_names = train_ds.class_names
print(f"\n🏷️ Detected Classes ({len(class_names)}): {class_names}")

# High performance pipeline with caching and background GPU prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("✅ Datasets successfully cached & configured with AUTOTUNE prefetching!")

In [ ]:
# ==============================================================================
# CELL 9: Show 3x3 Grid of Sample Images with Labels
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 10))

# Retrieve first batch from training dataset
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        img_array = images[i].numpy().astype("uint8")
        plt.imshow(img_array)
        
        class_idx = np.argmax(labels[i].numpy())
        class_name = class_names[class_idx]
        
        plt.title(f"{class_name}", fontsize=11, fontweight="bold", color='#15803d')
        plt.axis("off")

plt.suptitle("Sample Arecanut Training Images with Ground-Truth Labels", fontsize=14, fontweight="bold", y=0.95)
plt.tight_layout()
plt.show()